# Composite Vertical Profiles for O3, T, and U

In this notebook, we:
1. **Read** the multi-year merged NetCDF file to obtain a **climatology** (annual mean cycle) for each variable.
2. **Extract** a single year's data from the merged file (the 'reference' year), then compute **(YearData - Climatology)**.
3. **Extract** each experiment's data (e.g., `0008Feb_small`), then also compute **(Experiment - Climatology)**.
4. **Plot** these anomalies in a composite figure, including all experiments plus the reference. The X-axis (months) is trimmed to start from the latest start month among all runs and ends in May.

**Key points:**
- O3 is averaged over longitude and lat 60°–90°N (cosine-weighted).
- T is averaged over longitude and then we take the minimum from lat 60°–90°N.
- U is averaged over longitude and then we pick the value at lat=60°N (nearest).
- Each subplot's data is already `Data - Climatology`, so the plotting function only needs to plot these anomalies directly.


In [ ]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import os
import math

# ----------------------------------------------------
# Global paths and file references
# ----------------------------------------------------
MERGED_FILE = '/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.merged.nc'

# ----------------------------------------------------
# Data Processing Functions
# ----------------------------------------------------
def process_variable(ds, var):
    """
    Apply the specific transformation for O3, T, U:
    - O3: cos-weighted average over lat 60–90°N.
    - T: minimum over lat 60–90°N (after lon-mean).
    - U: pick lat=60°N (after lon-mean).
    """
    if var == 'O3':
        da = ds['O3'].mean(dim='lon')
        if 'member' in da.dims:
            da = da.mean(dim='member')
        da = da.sel(lat=slice(60, 90))
        weights = np.cos(np.deg2rad(da.lat))
        da = da.weighted(weights).mean(dim='lat')
    elif var == 'T':
        da = ds['T'].mean(dim='lon')
        if 'member' in da.dims:
            da = da.mean(dim='member')
        da = da.sel(lat=slice(60, 90)).min(dim='lat')
    elif var == 'U':
        da = ds['U'].mean(dim='lon')
        if 'member' in da.dims:
            da = da.mean(dim='member')
        da = da.sel(lat=60, method='nearest')
    else:
        raise ValueError("Unsupported variable: " + var)
    return da

def create_monthly_means(da, start_month=1, end_month=12):
    """
    Select time from start_month to end_month, then group by month,
    compute mean over time, and create a dummy time coordinate (2000-XX-15).
    Returns (time_coord, plev_vals, data_array) with shape (plev, month).
    """
    # Filter by month range
    da = da.sel(time=da.time.dt.month.isin(range(start_month, end_month+1)))

    # Group by month
    da_month = da.groupby('time.month').mean(dim='time')
    da_month = da_month.sortby('month')

    # Create dummy time coordinate
    months = da_month['month'].values
    dummy_time = np.array([np.datetime64(f'2000-{int(m):02d}-15') for m in months])

    # Rename 'month' -> 'time' and reorder dims
    da_month = da_month.rename({'month': 'time'})
    da_month = da_month.assign_coords(time=dummy_time)
    da_month = da_month.transpose('plev', 'time')

    plev_vals = da_month['plev'].values
    time_vals = da_month['time'].values
    data_array = da_month.values
    return time_vals, plev_vals, data_array

def read_climatology(var):
    """
    Read the merged file for all years (Jan–Dec), process the variable, then compute
    the multi-year monthly mean ("climatology").
    Returns an xarray DataArray with dims (plev, month), where 'month' is 1..12.
    """
    ds = xr.open_dataset(MERGED_FILE)
    da = process_variable(ds, var)
    ds.close()

    # We'll create monthly means for the entire year (1–12)
    # So that we can subtract them from any partial-year data.
    time_vals, plev_vals, data_arr = create_monthly_means(da, 1, 12)

    # Convert to xarray for easy alignment
    clim_da = xr.DataArray(
        data_arr,
        coords={'plev': plev_vals, 'time': time_vals},
        dims=['plev', 'time']
    )
    return clim_da

def read_year_data(var, year, start_month=1, end_month=12):
    """
    From the merged file, select only the specified 'year', process the variable,
    then create monthly means for start_month..end_month.
    Returns an xarray DataArray with dims (plev, time).
    """
    ds = xr.open_dataset(MERGED_FILE)
    ds_year = ds.sel(time=ds.time.dt.year == int(year))
    da = process_variable(ds_year, var)
    ds.close()

    time_vals, plev_vals, data_arr = create_monthly_means(da, start_month, end_month)
    year_da = xr.DataArray(
        data_arr,
        coords={'plev': plev_vals, 'time': time_vals},
        dims=['plev', 'time']
    )
    return year_da

def read_experiment_data(var, file_pattern, start_month=1, end_month=5):
    """
    Open the experiment file(s), process the variable, then create monthly means
    for the range [start_month..end_month].
    Returns an xarray DataArray with dims (plev, time).
    """
    ds = xr.open_mfdataset(file_pattern, concat_dim='member', combine='nested')
    da = process_variable(ds, var)
    ds.close()

    time_vals, plev_vals, data_arr = create_monthly_means(da, start_month, end_month)
    exp_da = xr.DataArray(
        data_arr,
        coords={'plev': plev_vals, 'time': time_vals},
        dims=['plev', 'time']
    )
    return exp_da

def get_anomaly(data_da, clim_da):
    """
    Compute anomaly = data_da - clim_da, aligning by month.
    data_da, clim_da both have dims (plev, time) with time as dummy dates.
    We can align by the month extracted from 'time'.
    """
    # Convert time coords to month
    data_months = [int(str(t)[5:7]) for t in data_da.time.values]
    # For clim_da, we also do the same
    clim_months = [int(str(t)[5:7]) for t in clim_da.time.values]
    # We'll create a copy of clim_da that only includes months present in data_da
    # Then do the subtraction.
    aligned_vals = np.zeros_like(data_da.values)
    for i, m in enumerate(data_months):
        # find the index in clim_da that has month == m
        # because we used dummy times like 2000-mm-15
        j = clim_months.index(m)
        aligned_vals[:, i] = data_da.values[:, i] - clim_da.values[:, j]

    anom_da = xr.DataArray(
        aligned_vals,
        coords={'plev': data_da.plev, 'time': data_da.time},
        dims=['plev', 'time']
    )
    return anom_da

# ----------------------------------------------------
# Plotting Functions
# ----------------------------------------------------
def auto_subplot_layout(n):
    """Return (rows, cols) for n subplots, using a square-like layout."""
    cols = int(math.ceil(math.sqrt(n)))
    rows = int(math.ceil(n / cols))
    return rows, cols

def determine_common_months(runs):
    """
    Among all runs' time coords, find the maximum start month.
    Then define months from that start month to 5 (May).
    """
    if not runs:
        return [3, 4, 5]
    start_months = []
    for run in runs:
        if len(run['time']) > 0:
            m = int(str(run['time'][0])[5:7])  # e.g., '2000-03-15' -> 03
            start_months.append(m)
    if not start_months:
        return [3, 4, 5]
    latest_start = max(start_months)
    return list(range(latest_start, 6))  # from latest_start to 5

def trim_to_months(run, months):
    """Trim the run['time'] and run['data'] to only include specified months."""
    if not months:
        return run
    keep_indices = []
    for i, t in enumerate(run['time']):
        m = int(str(t)[5:7])
        if m in months:
            keep_indices.append(i)
    run['time'] = run['time'][keep_indices]
    run['data'] = run['data'][:, keep_indices]
    return run

def plot_composite(runs, ref_run, var, composite_name, year, save_path):
    """
    Create subplots for each run (already in anomaly form) + one subplot for REF.
    The color scale is unified across all subplots.
    The X-axis is from the maximum start month among all runs to May.
    """
    all_runs = runs + [ref_run]

    # 1) Determine common months
    common_months = determine_common_months(all_runs)

    # 2) Trim each run to these months
    for r in all_runs:
        trim_to_months(r, common_months)

    # 3) Collect all anomalies for color scale
    all_anom = []
    for r in all_runs:
        all_anom.append(r['data'].ravel())
    all_anom = np.concatenate(all_anom)
    vmax = np.nanmax(np.abs(all_anom))
    vmin = -vmax

    # 4) Set up subplots
    total_subplots = len(all_runs)
    rows, cols = auto_subplot_layout(total_subplots)
    fig, axes = plt.subplots(rows, cols, figsize=(4*cols, 3*rows), squeeze=False)

    # Prepare x-axis tick positions
    # We'll just take the reference run's time as baseline for labeling,
    # or the first run's time if reference is empty.
    subplot_idx = 0
    cf = None

    for run in all_runs:
        r = subplot_idx // cols
        c = subplot_idx % cols
        ax = axes[r][c]

        if len(run['time']) > 0:
            time_mpl = mdates.date2num(run['time'])
            cf = ax.contourf(time_mpl, run['plev'], run['data'], levels=20,
                             cmap='RdBu_r', vmin=vmin, vmax=vmax)
            ax.set_yscale('log')
            ax.invert_yaxis()
            ax.set_xlim(time_mpl[0], time_mpl[-1])

            # Format X ticks
            ax.xaxis.set_major_locator(mdates.MonthLocator())
            ax.xaxis.set_major_formatter(mdates.DateFormatter('%b'))

        ax.set_ylabel('Pressure (Pa)', fontsize=8)
        ax.set_title(f"{run['label']} (Year {run['year']})", fontsize=10)
        subplot_idx += 1

    # Hide unused subplots
    for i in range(subplot_idx, rows*cols):
        r = i // cols
        c = i % cols
        axes[r][c].axis('off')

    # Add a colorbar
    if cf is not None:
        cax = fig.add_axes([0.92, 0.15, 0.02, 0.7])
        fig.colorbar(cf, cax=cax, label=f'{var} Anomaly')

    fig.suptitle(f"{composite_name} - {var} Anomaly\nYear {year} (Latest start month to May)", fontsize=14)
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f"[INFO] Saved composite figure: {save_path}")

# ----------------------------------------------------
# Define composite groups (20 groups) and generate plots
# ----------------------------------------------------
# Helper to build file paths for small pert & nocouple
def get_year_small_pert(year):
    base = f'/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_{year}'
    file_Feb = os.path.join(base, 'Feb', f'BWCN.e122.f19_g16.002.{year}-02.*.nc')
    file_Mar = os.path.join(base, 'Mar', f'BWCN.e122.f19_g16.002.{year}-03.*.nc')
    return file_Feb, file_Mar

def get_year_nocouple(year):
    file_Feb = f'/home/weiji/restart_exam/nocouple_data/{year}/Feb_NOCOUPL/{year}-02/*.nc'
    file_Mar = f'/home/weiji/restart_exam/nocouple_data/{year}/Mar_NOCOUPL/{year}-03/*.nc'
    return file_Feb, file_Mar

# Predefine known file patterns for 0008
file_0008_Jan_small = '/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008/Jan/BWCN.e122.f19_g16.002.0008-01.*.nc'
file_0008_Feb_small = '/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008/Feb/BWCN.e122.f19_g16.002.0008-02.*.nc'
file_0008_Mar_small = '/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008/Mar/BWCN.e122.f19_g16.002.0008-03.*.nc'
file_0008_Feb_large = '/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008/Feb_2/BWCN.e122.f19_g16.002.0008-02.*.nc'
file_0008_Mar_large = '/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008/Mar_2/BWCN.e122.f19_g16.002.0008-03.*.nc'
file_0008_Apr_large = '/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008/Apr/BWCN.e122.f19_g16.002.0008-04.*.nc'
file_0008_Feb_moist = '/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008/Feb_3/BWCN.e122.f19_g16.002.0008-02.*.nc'
file_0008_Mar_moist = '/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008/Mar_3/BWCN.e122.f19_g16.002.0008-03.*.nc'

composite_groups = [
    # 1
    {'name': 'Composite 1', 'year': '0008', 'runs': [
        {'label': 'Jan small pertlim', 'file': file_0008_Jan_small},
        {'label': 'Feb small pertlim', 'file': file_0008_Feb_small},
        {'label': 'Mar small pertlim', 'file': file_0008_Mar_small}
    ]},
    # 2
    {'name': 'Composite 2', 'year': '0008', 'runs': [
        {'label': 'Feb large pertlim', 'file': file_0008_Feb_large},
        {'label': 'Mar large pertlim', 'file': file_0008_Mar_large},
        {'label': 'Apr large pertlim', 'file': file_0008_Apr_large}
    ]},
    # 3
    {'name': 'Composite 3', 'year': '0008', 'runs': [
        {'label': 'Feb small pertlim', 'file': file_0008_Feb_small},
        {'label': 'Feb large pertlim', 'file': file_0008_Feb_large},
        {'label': 'Feb moist pertlim', 'file': file_0008_Feb_moist}
    ]},
    # 4
    {'name': 'Composite 4', 'year': '0008', 'runs': [
        {'label': 'Mar small pertlim', 'file': file_0008_Mar_small},
        {'label': 'Mar large pertlim', 'file': file_0008_Mar_large},
        {'label': 'Mar moist pertlim', 'file': file_0008_Mar_moist}
    ]},
    # 5
    {'name': 'Composite 5', 'year': '0003', 'runs': [
        {'label': 'Feb small pertlim', 'file': get_year_small_pert('0003')[0]},
        {'label': 'Mar small pertlim', 'file': get_year_small_pert('0003')[1]}
    ]},
    # 6
    {'name': 'Composite 6', 'year': '0013', 'runs': [
        {'label': 'Feb small pertlim', 'file': get_year_small_pert('0013')[0]},
        {'label': 'Mar small pertlim', 'file': get_year_small_pert('0013')[1]}
    ]},
    # 7
    {'name': 'Composite 7', 'year': '0014', 'runs': [
        {'label': 'Feb small pertlim', 'file': get_year_small_pert('0014')[0]},
        {'label': 'Mar small pertlim', 'file': get_year_small_pert('0014')[1]}
    ]},
    # 8
    {'name': 'Composite 8', 'year': '0019', 'runs': [
        {'label': 'Feb small pertlim', 'file': get_year_small_pert('0019')[0]},
        {'label': 'Mar small pertlim', 'file': get_year_small_pert('0019')[1]}
    ]},
    # 9
    {'name': 'Composite 9', 'year': '0008', 'runs': [
        {'label': 'Feb nocouple', 'file': '/home/weiji/restart_exam/nocouple_data/0008/Feb_NOCOUPL/0008-02/*.nc'},
        {'label': 'Mar nocouple', 'file': '/home/weiji/restart_exam/nocouple_data/0008/Mar_NOCOUPL/0008-03/*.nc'}
    ]},
    # 10
    {'name': 'Composite 10', 'year': '0013', 'runs': [
        {'label': 'Feb nocouple', 'file': '/home/weiji/restart_exam/nocouple_data/0013/Feb_NOCOUPL/0013-02/*.nc'},
        {'label': 'Mar nocouple', 'file': '/home/weiji/restart_exam/nocouple_data/0013/Mar_NOCOUPL/0013-03/*.nc'}
    ]},
    # 11
    {'name': 'Composite 11', 'year': '0014', 'runs': [
        {'label': 'Feb nocouple', 'file': '/home/weiji/restart_exam/nocouple_data/0014/Feb_NOCOUPL/0014-02/*.nc'},
        {'label': 'Mar nocouple', 'file': '/home/weiji/restart_exam/nocouple_data/0014/Mar_NOCOUPL/0014-03/*.nc'}
    ]},
    # 12
    {'name': 'Composite 12', 'year': '0019', 'runs': [
        {'label': 'Feb nocouple', 'file': '/home/weiji/restart_exam/nocouple_data/0019/Feb_NOCOUPL/0019-02/*.nc'},
        {'label': 'Mar nocouple', 'file': '/home/weiji/restart_exam/nocouple_data/0019/Mar_NOCOUPL/0019-03/*.nc'}
    ]},
    # 13
    {'name': 'Composite 13', 'year': '0008', 'runs': [
        {'label': 'Feb small pertlim', 'file': file_0008_Feb_small},
        {'label': 'Feb nocouple', 'file': '/home/weiji/restart_exam/nocouple_data/0008/Feb_NOCOUPL/0008-02/*.nc'}
    ]},
    # 14
    {'name': 'Composite 14', 'year': '0008', 'runs': [
        {'label': 'Mar small pertlim', 'file': file_0008_Mar_small},
        {'label': 'Mar nocouple', 'file': '/home/weiji/restart_exam/nocouple_data/0008/Mar_NOCOUPL/0008-03/*.nc'}
    ]},
    # 15
    {'name': 'Composite 15', 'year': '0013', 'runs': [
        {'label': 'Feb small pertlim', 'file': get_year_small_pert('0013')[0]},
        {'label': 'Feb nocouple', 'file': '/home/weiji/restart_exam/nocouple_data/0013/Feb_NOCOUPL/0013-02/*.nc'}
    ]},
    # 16
    {'name': 'Composite 16', 'year': '0013', 'runs': [
        {'label': 'Mar small pertlim', 'file': get_year_small_pert('0013')[1]},
        {'label': 'Mar nocouple', 'file': '/home/weiji/restart_exam/nocouple_data/0013/Mar_NOCOUPL/0013-03/*.nc'}
    ]},
    # 17
    {'name': 'Composite 17', 'year': '0014', 'runs': [
        {'label': 'Feb small pertlim', 'file': get_year_small_pert('0014')[0]},
        {'label': 'Feb nocouple', 'file': '/home/weiji/restart_exam/nocouple_data/0014/Feb_NOCOUPL/0014-02/*.nc'}
    ]},
    # 18
    {'name': 'Composite 18', 'year': '0014', 'runs': [
        {'label': 'Mar small pertlim', 'file': get_year_small_pert('0014')[1]},
        {'label': 'Mar nocouple', 'file': '/home/weiji/restart_exam/nocouple_data/0014/Mar_NOCOUPL/0014-03/*.nc'}
    ]},
    # 19
    {'name': 'Composite 19', 'year': '0019', 'runs': [
        {'label': 'Feb small pertlim', 'file': get_year_small_pert('0019')[0]},
        {'label': 'Feb nocouple', 'file': '/home/weiji/restart_exam/nocouple_data/0019/Feb_NOCOUPL/0019-02/*.nc'}
    ]},
    # 20
    {'name': 'Composite 20', 'year': '0019', 'runs': [
        {'label': 'Mar small pertlim', 'file': get_year_small_pert('0019')[1]},
        {'label': 'Mar nocouple', 'file': '/home/weiji/restart_exam/nocouple_data/0019/Mar_NOCOUPL/0019-03/*.nc'}
    ]}
]

def generate_all_composites():
    output_dir = '/home/weiji/restart_exam/plots/Vertical_Profiles_new/'
    variables = ['O3', 'T', 'U']

    for var in variables:
        print(f"\n[INFO] Generating composite plots for {var}...")
        # 1) Read the global climatology (1–12 months)
        clim_da = read_climatology(var)  # dims: (plev, time)

        for group in composite_groups:
            year = group['year']
            composite_name = group['name']
            runs_info = group['runs']

            # 2) Read single-year data from merged (for REF), subtract climatology
            year_da = read_year_data(var, year, 1, 12)  # entire year if needed
            ref_anom = get_anomaly(year_da, clim_da)
            ref_run = {
                'label': 'REF',
                'year': year,
                'time': ref_anom.time.values,
                'plev': ref_anom.plev.values,
                'data': ref_anom.values
            }

            # 3) Read each experiment, subtract climatology
            exp_runs = []
            for r in runs_info:
                exp_da = read_experiment_data(var, r['file'], 1, 5)
                exp_anom = get_anomaly(exp_da, clim_da)
                exp_runs.append({
                    'label': r['label'],
                    'year': year,
                    'time': exp_anom.time.values,
                    'plev': exp_anom.plev.values,
                    'data': exp_anom.values
                })

            # 4) Plot them all
            save_path = os.path.join(output_dir, f"{composite_name.replace(' ', '_')}_{var}.png")
            plot_composite(exp_runs, ref_run, var, composite_name, year, save_path)

    print("\n[INFO] All composite plots have been generated.")
generate_all_composites()
# Uncomment to run in a real environment
# generate_all_composites()
# After running generate_all_composites(), you should find 20 composites x 3 variables = 60 images.
